# Hard Case: Lazy-load / Infinite Scroll

Halaman *infinite scroll* memuat data baru saat kita scroll ke bawah (mis. feed sosial media,
katalog produk). Ada **dua pendekatan**:

- **Pendekatan A (disarankan): cari hidden API.** Halaman scroll biasanya memanggil sebuah
  endpoint JSON di belakang layar. Buka **DevTools → Network → XHR**, temukan endpoint-nya,
  lalu panggil langsung dengan `requests`. **Cepat, ringan, bersih.**
- **Pendekatan B: Selenium scroll.** Kalau tidak ada API yang bisa dipakai, kita scroll
  pakai browser sampai data berhenti bertambah.

Latihan pakai `https://quotes.toscrape.com/scroll` (yang di belakang memanggil
`https://quotes.toscrape.com/api/quotes?page=N`).

**Tooling:** `requests` (cara A), `selenium` (cara B).


## Pendekatan A — Hidden API (paling efisien)

Endpoint `…/api/quotes?page=N` mengembalikan JSON dengan field `has_next`. Kita loop
halaman demi halaman sampai `has_next == False`. Tidak perlu browser sama sekali.


In [1]:
import requests

API = "https://quotes.toscrape.com/api/quotes"


def ambil_via_api():
    semua = []
    page = 1
    while True:
        r = requests.get(API, params={"page": page}, timeout=10)
        r.raise_for_status()
        data = r.json()
        for q in data["quotes"]:
            semua.append(
                {"text": q["text"], "author": q["author"]["name"], "tags": q["tags"]}
            )
        print(f"page {page}: +{len(data['quotes'])}  (has_next={data['has_next']})")
        if not data["has_next"]:  # tidak ada halaman berikutnya -> selesai
            break
        page += 1
    return semua


hasil_api = ambil_via_api()
print("\nTotal quote (via API):", len(hasil_api))
hasil_api[0]


page 1: +10  (has_next=True)


page 2: +10  (has_next=True)


page 3: +10  (has_next=True)


page 4: +10  (has_next=True)


page 5: +10  (has_next=True)


...

page 7: +10  (has_next=True)


page 8: +10  (has_next=True)


page 9: +10  (has_next=True)


page 10: +10  (has_next=False)

Total quote (via API): 100


{'text': '“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”',
 'author': 'Albert Einstein',
 'tags': ['change', 'deep-thoughts', 'thinking', 'world']}

## Pendekatan B — Selenium scroll

Kalau tidak ada API, kita scroll halaman pakai browser. Polanya: scroll ke paling bawah →
beri jeda agar konten baru dimuat → hitung elemen. **Berhenti ketika jumlah elemen tidak
bertambah lagi** (artinya sudah habis).


In [2]:
import time

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By


def buat_driver(headless=True):
    o = Options()
    if headless:  # set False kalau mau lihat scroll-nya
        o.add_argument("--headless=new")
    o.add_argument("--window-size=1280,900")
    return webdriver.Chrome(options=o)


driver = buat_driver()
try:
    driver.get("https://quotes.toscrape.com/scroll")
    jumlah_sebelumnya = 0
    while True:
        # scroll ke paling bawah halaman
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1)  # beri waktu konten baru dimuat (lazy-load)

        quotes = driver.find_elements(By.CSS_SELECTOR, ".quote")
        if len(quotes) == jumlah_sebelumnya:  # tidak nambah -> sudah habis
            break
        jumlah_sebelumnya = len(quotes)
        print("quote setelah scroll:", jumlah_sebelumnya)

    print("\nTotal quote (via Selenium scroll):", jumlah_sebelumnya)
finally:
    driver.quit()


quote setelah scroll: 20


quote setelah scroll: 30


quote setelah scroll: 40


quote setelah scroll: 50


quote setelah scroll: 60


quote setelah scroll: 70


quote setelah scroll: 80


quote setelah scroll: 90


quote setelah scroll: 100



Total quote (via Selenium scroll): 100


## Kesimpulan & Latihan

- **Selalu cek Network tab dulu** untuk hidden API — biasanya jauh lebih cepat & stabil
  daripada scroll pakai Selenium.
- Pola berhenti scroll: bandingkan jumlah elemen sebelum vs sesudah; sama → berhenti.
  (Alternatif lebih canggih: bandingkan `document.body.scrollHeight`.)
- Dua pendekatan ini harus memberi **jumlah quote yang sama**.

**Latihan:** masukkan hasil API ke `pandas.DataFrame`, lalu cari 5 tag paling sering muncul.


In [3]:
# Contoh jawaban latihan
import pandas as pd

df = pd.DataFrame(hasil_api)
print("Bentuk DataFrame:", df.shape)

# tags adalah list per baris -> explode jadi satu tag per baris, lalu hitung
top_tags = df.explode("tags")["tags"].value_counts().head(5)
print("\n5 tag terpopuler:")
print(top_tags)


Bentuk DataFrame: (100, 3)

5 tag terpopuler:
tags
love             14
inspirational    13
life             13
humor            12
books            11
Name: count, dtype: int64
